In [3]:
import pandas as pd
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt

## Processing Potentiometer Data

Let's first read the data that are relevant to organize the voltage level according to the time that has passed.

In [5]:
# Step 1: Read data
# Identify potential value
ec_df = pd.read_csv('./data/Coated_SS-052125_800mV_vs_Ag-AgCl/Coated_SS-052125_800mV_vs_Ag-AgCl.txt', header=None)

FileNotFoundError: [Errno 2] No such file or directory: './data/Example/Coated_SS-052125_800mV_vs_Ag-AgCl/Coated_SS-052125_800mV_vs_Ag-AgCl.txt'

In [ ]:

# Clean data
row_index = ec_df[ec_df.apply(lambda row: row.astype(str).str.contains('Time/sec').any(), axis=1)].index[0]

ec_df_aux_data = ec_df.iloc[:row_index]

ec_df_cleaned = ec_df.iloc[row_index+1:]

ec_df_cleaned.columns = ['Time/sec', 'Current/A']
ec_df_cleaned.set_index('Time/sec', inplace=True)

ec_df_cleaned['Current/A'] = ec_df_cleaned['Current/A'].astype(float)
ec_df_cleaned

,Time/sec,Current/A
0,0.000,-0.000096
1,0.128,-0.000087
2,0.256,-0.000082
3,0.384,-0.000079
4,0.512,-0.000076
...,...,...
221101,28300.000,-0.000081
221102,28300.000,-0.000081
221103,28300.000,-0.000081
221104,28300.000,-0.000081


In [ ]:
ec_df = ec_df_cleaned

### Defining Intervals

We use a Hashmap structure to associate each time (minutes) $t$ with some voltage (volts) $v$ of size $N$ in the dictionary $t_{N}$. 

Voltages are stored in an array structure in $V$ and repeated $\text{cycles}$ times to replicate numerous consecutive tests.

The first interval is defined by $I_0$ and will increment by period $T$ until the current number of entries  $n \geq N$, which is the size of $V$.

$V_q$ will be a First-In Last-Out (FILO) queue to ensure the cyclical behavior of the period through the following logic:

* if $t_N$ is empty, fill the first entry with the first voltage entry $V[0]$
* if there are no voltages, fill it with the predetermined voltages $V$
* Otherwise, indicate any given key in $t_N$ using $I_t$ which is updated in every pass to be $T$ more than the last. The value associated with $I_t$ will be the left-most value in $V_q$ 


In [ ]:
# Step 2: Organize the intrevals
I0 = 1800
T = 3600
It = I0

cycles = 3 # change cycle number 
V = [-0.4, 0, 0.4]*cycles # change array for desired set of voltages


# Start a queue
Vq = deque()


n = 0
N = len(V)

tN = {} # time associated mapped to voltage

while(n < N):
    if not tN:
        tN[I0] = V[0]
        continue
    if not Vq:
        Vq = deque(V)

    tN[It] = Vq.popleft()
    It += T
    n += 1
    

tN

{1800: -0.4,
 5400: 0,
 9000: 0.4,
 12600: -0.4,
 16200: 0,
 19800: 0.4,
 23400: -0.4,
 27000: 0,
 30600: 0.4}

Now, we're ready to associate each interval and value inside our given data.

In [71]:
# Step 3: Replace intervals at appropriate sections
ec_df['Voltage'] = np.nan

for t in sorted(tN):  # ensure ascending order
    mask = (ec_df['Time/sec'] <= t) & (ec_df['Voltage'].isna())
    ec_df.loc[mask, 'Voltage'] = tN[t]


The different sectioned voltages are shown below:

In [72]:

prev_time = 0
for key in tN.keys():
    mask = (prev_time < ec_df['Time/sec']) & (ec_df['Time/sec'] <= key)
    print(
        ec_df[mask].head(), 
        ec_df[mask].tail(), 
        sep=".\n.\n.\n.\n"
    )

    prev_time = key

   Time/sec   Current/A  Voltage
1     0.128   -0.000087     -0.4
2     0.256   -0.000082     -0.4
3     0.384   -0.000079     -0.4
4     0.512   -0.000076     -0.4
5     0.640   -0.000074     -0.4.
.
.
.
       Time/sec   Current/A  Voltage
14093    1800.0   -0.000005     -0.4
14094    1800.0   -0.000005     -0.4
14095    1800.0   -0.000005     -0.4
14096    1800.0   -0.000005     -0.4
14097    1800.0   -0.000005     -0.4
       Time/sec   Current/A  Voltage
14098    1810.0   -0.000005      0.0
14099    1810.0   -0.000005      0.0
14100    1810.0   -0.000005      0.0
14101    1810.0   -0.000005      0.0
14102    1810.0   -0.000005      0.0.
.
.
.
       Time/sec   Current/A  Voltage
42218    5400.0    0.000015      0.0
42219    5400.0    0.000015      0.0
42220    5400.0    0.000015      0.0
42221    5400.0    0.000015      0.0
42222    5400.0    0.000015      0.0
       Time/sec   Current/A  Voltage
42223    5410.0    0.000014      0.4
42224    5410.0    0.000014      0.4
42225    54

## Cleaning ICP-MS Data


To align these data we take into account three externally measured variables: 

* Potential Calibration of the Reference Electrode (PCRE) with respect to Reversible Hydrogen Electrode (RHE): `potent_calib`

* the difference between the E-Chem and the ICP-MS start times: `del_start`

* the Response Delay which indicates the difference in which some electrochemical event occurs and when it is recorded within the ICP-MS: `response_delay`


To begin, let's clean the EChem data and offset the echem data by the PCRE and store it in its respective array.

In [73]:
''' 
Values used in Matt Sweers' parameter template file
'''

potent_calib = .268 
del_start = 108
response_delay = 23

In [74]:
# Clean the data of white space
ec_df_clean = ec_df.rename(columns=lambda x: x.strip())
ec_df_clean = ec_df_clean.sort_values("Time/sec")

In [75]:
ec_cal = ec_df_clean.copy()
ec_cal['Current/A'] = ec_cal['Current/A'] + potent_calib  # Apply calibration

ec_cal_np = ec_cal.to_numpy()  # Convert calibrated dataset into NumPy


In [76]:
icp_df = pd.read_csv('./data/400_mV_041725/ICP_MS.csv')

ICP-MS and E-Chem often exhibit different shapes as the data are collected in a distinct fashions. For example in `400_mV_041725.csv`, the ICP-MS dataset is of shape ($\text{columns} \times \text{rows}$) $90711 \times 6$ whereas the E-Chem dataframes is $221106 \times 3$. This necessitates truncation of the larger dataset to best fit the smaller counterpart.

In addition, we offset the time values by the appropriate amount by subtracting the sum of the start and response delay `del_start` + `response_delay`.

In [77]:
# Cut off ICP-MS data before the E-Chem data begins population
icp_trunc = icp_df[icp_df['Time'] >(del_start+response_delay)]

# Shift time so that it starts from zero relative to E-Chem
icp_trunc['Time'] = icp_trunc['Time']-(del_start+response_delay)

# Keep a NumPy copy
icp_trunc_np = icp_trunc.to_numpy()

/var/folders/vm/_ljljg3d2ln_x5m_sd3t34kr0000gn/T/ipykernel_32772/3466582724.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  icp_trunc['Time'] = icp_trunc['Time']-(del_start+response_delay)


In [78]:
icp_trunc

,Replicate,Reading,Time,Cr -/52(Pulse/CPS),Mn -/55(Pulse/CPS),Fe -/56(Pulse/CPS),Ni -/60(Pulse/CPS),Mo -/98(Pulse/CPS)
405,1,406,0.224,36065.467740,3200.358440,492836.93740,11764.842410,6141.319770
406,1,407,0.548,37870.128690,3440.414226,539622.82640,12045.075790,6261.371867
407,1,408,0.872,37649.546800,3920.537898,592372.20500,12825.754920,7161.794746
408,1,409,1.196,40938.574910,3760.494881,660232.13500,15007.879140,6421.442898
409,1,410,1.520,42242.362400,3860.521556,720229.13810,14267.120720,6981.705631
...,...,...,...,...,...,...,...,...
90706,1,90707,29258.832,160.000896,20.000014,13166.06429,1680.098790,20.000014
90707,1,90708,29259.156,140.000686,0.000000,15007.87914,2100.154361,20.000014
90708,1,90709,29259.480,220.001694,40.000056,16649.69678,1900.126358,0.000000
90709,1,90710,29259.804,220.001694,20.000014,15268.15472,2640.243959,20.000014


# Interpolation
The `final_dataset` will indicate the result of the last two steps and this one: interpolation. The goal is to linearly interpret values $t_1, t_2$, the times before and after any given ICP-MS datapoint being mapped to the E-Chem data defined as $t$. The interpolated potential $\left( E\{t \} \right)$ and current density $\left(i\{t \}\right)$ at $t$ can be obtained the following two equations:

$$E(t) = E_0 + \left( \frac{t-t_0}{t_1 - t_0} \right) \left(E_1 - E_0\right)$$

$$i(t) = i_0 + \left( \frac{t-t_0}{t_1 - t_0} \right) \left(i_1 - i_0\right)$$

In [79]:
# Assume icp_trunc has columns: ['Time', 'ICP_MS_Signal']
ec_times = ec_cal['Time/sec'].values
ec_voltages = ec_cal['Voltage'].values
ec_currents = ec_cal['Current/A'].values

fin = []

for i in range(len(icp_trunc)):
    fin_t = icp_trunc.iloc[i]['Time']

    idx = np.searchsorted(ec_times, fin_t)

    if idx == 0:
        interp_voltage = ec_voltages[0]
        interp_current = ec_currents[0]
    elif idx >= len(ec_times):
        interp_voltage = ec_voltages[-1]
        interp_current = ec_currents[-1]
    else:
        t0, t1 = ec_times[idx - 1], ec_times[idx]
        v0, v1 = ec_voltages[idx - 1], ec_voltages[idx]
        c0, c1 = ec_currents[idx - 1], ec_currents[idx]

        interp_voltage = v0 + (fin_t - t0) * (v1 - v0) / (t1 - t0)
        interp_current = c0 + (fin_t - t0) * (c1 - c0) / (t1 - t0)

    fin.append([fin_t, interp_voltage, interp_current])

fin_df = pd.DataFrame(fin, columns=['Time', 'Potential/v', 'Current/A_(interp\'d)'])

# Drop unwanted columns from icp_trunc and reset index
extra_cols = icp_trunc.drop(columns=['Replicate', 'Reading', 'Time'], errors='ignore').reset_index(drop=True)

# Concatenate the two DataFrames column-wise
fin_df = pd.concat([extra_cols, fin_df], axis=1)

fin_df = fin_df.set_index('Time')
fin_df



,Cr -/52(Pulse/CPS),Mn -/55(Pulse/CPS),Fe -/56(Pulse/CPS),Ni -/60(Pulse/CPS),Mo -/98(Pulse/CPS),Potential/v,Current/A_(interp'd)
Time,,,,,,,
0.224,36065.467740,3200.358440,492836.93740,11764.842410,6141.319770,-0.4,0.267917
0.548,37870.128690,3440.414226,539622.82640,12045.075790,6261.371867,-0.4,0.267924
0.872,37649.546800,3920.537898,592372.20500,12825.754920,7161.794746,-0.4,0.267928
1.196,40938.574910,3760.494881,660232.13500,15007.879140,6421.442898,-0.4,0.267931
1.520,42242.362400,3860.521556,720229.13810,14267.120720,6981.705631,-0.4,0.267932
...,...,...,...,...,...,...,...
29258.832,160.000896,20.000014,13166.06429,1680.098790,20.000014,0.4,0.267919
29259.156,140.000686,0.000000,15007.87914,2100.154361,20.000014,0.4,0.267919
29259.480,220.001694,40.000056,16649.69678,1900.126358,0.000000,0.4,0.267919


In [1]:
x = np.linspace(0,2,100)

t = fin_df.index
current = fin_df['Current/A_(interp\'d)']
potential = fin_df['Potential/v']

fig, ax1 = plt.subplots(figsize=(10, 7), layout='constrained')


color = 'tab:red'
ax1.set_xlabel('t (seconds)')
ax1.plot(t, current, label='Current', color=color)
ax1.set_ylabel('Current (amps)')

color = 'tab:blue'
ax2 = ax1.twinx()
ax2.set_ylabel('Potential (v)')
ax2.plot(t, potential, label='Potential', color=color)

fig.tight_layout()
ax1.legend()
ax2.legend()


NameError: name 'np' is not defined